In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
import os


DRIVE_ROOT = '/content/drive/MyDrive/librimix_3000'

INPUT_FOLDERS = {
    'mix_both': 'mix_both',
    's1': 's1',
    's1_noisy_40': 's1_noisy_40',
}
OUTPUT_FOLDERS = {
    'after_reuse_mix': 'after_reuse_mix',
    'after_reuse_s1': 'after_reuse_s1',
    'after_reuse_s1_noisy_40': 'after_reuse_s1_noisy_40',
}

print("=== INPUTS ===")
for label, folder in INPUT_FOLDERS.items():
    path = os.path.join(DRIVE_ROOT, folder)
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.endswith('.wav')])
        print(f"  {label}: {n} files")
    else:
        print(f"  {label}: NOT FOUND at {path} — upload it!")

print("\n=== OUTPUTS (already processed) ===")
for label, folder in OUTPUT_FOLDERS.items():
    path = os.path.join(DRIVE_ROOT, folder)
    os.makedirs(path, exist_ok=True)
    n = len([f for f in os.listdir(path) if f.endswith('.wav')])
    print(f"  {label}: {n} files done")


In [ ]:
!pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
print("\n=== DONE. Now: Runtime -> Restart session, then SKIP this cell and run Cell 4b ===")


In [ ]:
import torch, sys, re, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("CUDA not available! Go to Runtime -> Change runtime type -> GPU")

!pip install -q huggingface_hub safetensors librosa soundfile

torch_ver = re.match(r'(\d+\.\d+)', torch.__version__).group(1)
py_ver = f"cp{''.join(str(x) for x in sys.version_info[:2])}"
cxx_abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"Detected: torch{torch_ver}, {py_ver}, cxx11abi{cxx_abi}")

def try_install_wheel(urls, fallback_pip):
    for url in urls:
        print(f"  Trying: {url.split('/')[-1]}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", url],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  OK!")
            return True
        print(f"  Failed (trying next...)")
    print(f"  No pre-built wheel found, building from source...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install"] + fallback_pip
    )
    return True

causal_versions = ["v1.6.1.post4", "v1.6.1.post1", "v1.5.0.post8"]
causal_urls = [
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/{v}/causal_conv1d-{v[1:]}+cu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
    for v in causal_versions
]
print("\nInstalling causal-conv1d:")
try_install_wheel(causal_urls, ["causal-conv1d>=1.4.0", "--no-build-isolation"])

mamba_urls = [
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
]
print("\nInstalling mamba-ssm:")
try_install_wheel(mamba_urls, ["mamba-ssm", "--no-build-isolation"])

import mamba_ssm
print(f"\nmamba-ssm version: {mamba_ssm.__version__}")
print("SUCCESS!")


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)
print("RE-USE downloaded.")


In [ ]:
import os, sys, math, torch, torch.nn as nn, torchaudio
import numpy as np
from tqdm import tqdm
from huggingface_hub import hf_hub_download

sys.path.insert(0, './REUSE')
from models.stfts import mag_phase_stft, mag_phase_istft
from models.generator_SEMamba_time_d4 import SEMamba
from utils.util import load_config, pad_or_trim_to_match

RELU = nn.ReLU()
device = torch.device('cuda')

config_path = hf_hub_download(repo_id='nvidia/RE-USE', filename='config.json')
cfg = load_config(config_path)
n_fft = cfg['stft_cfg']['n_fft']
hop_size = cfg['stft_cfg']['hop_size']
win_size = cfg['stft_cfg']['win_size']
compress_factor = cfg['model_cfg']['compress_factor']
sampling_rate = cfg['stft_cfg']['sampling_rate']

SE_model = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_model.eval()
print("RE-USE model loaded on GPU.")

def make_even(value):
    value = int(round(value))
    return value if value % 2 == 0 else value + 1

def enhance_short(wav_tensor, sr):
    wav = wav_tensor.to(device)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)
    with torch.no_grad():
        noisy_mag, noisy_pha, _ = mag_phase_stft(
            wav, n_fft=n_fft_s, hop_size=hop_s, win_size=win_s,
            compress_factor=compress_factor, center=True, addeps=False
        )
        amp_g, pha_g, _ = SE_model(noisy_mag, noisy_pha)
        mag = torch.expm1(RELU(amp_g))
        zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
        amp_g[:, :, (zero_portion > 0.5)[0]] = 0
        audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
        audio_g = pad_or_trim_to_match(wav.detach(), audio_g, pad_value=1e-8)
    return audio_g.cpu()

print("Enhancement function ready.")


In [ ]:
import os, time, torch
import soundfile as sf
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/librimix_3000'

RUNS = [
    ('mix_both', 'after_reuse_mix'),
    ('s1', 'after_reuse_s1'),
    ('s1_noisy_40', 'after_reuse_s1_noisy_40'),
]

def load_wav(path):
    data, sr = sf.read(path, dtype='float32')
    wav = torch.from_numpy(data)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    else:
        wav = wav.T
    return wav, sr

def save_wav(path, wav_tensor, sr):
    data = wav_tensor.numpy()
    if data.ndim == 2:
        data = data.T
    sf.write(path, data, sr)

all_errors = []

for input_folder, output_folder in RUNS:
    input_dir = os.path.join(DRIVE_ROOT, input_folder)
    output_dir = os.path.join(DRIVE_ROOT, output_folder)
    os.makedirs(output_dir, exist_ok=True)

    all_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.wav')])
    already_done = set(f for f in os.listdir(output_dir) if f.endswith('.wav'))
    remaining = [f for f in all_files if f not in already_done]

    print(f"\n{'='*60}")
    print(f"  {input_folder} → {output_folder}")
    print(f"  Total: {len(all_files)} | Done: {len(already_done)} | Remaining: {len(remaining)}")
    print(f"{'='*60}")

    if not remaining:
        print("  Skipping — all files already processed.")
        continue

    errors = []
    start_time = time.time()

    for i, fname in enumerate(tqdm(remaining, desc=output_folder)):
        try:
            wav, sr = load_wav(os.path.join(input_dir, fname))
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)
            enhanced = enhance_short(wav, sr)
            save_wav(os.path.join(output_dir, fname), enhanced, sr)
        except Exception as e:
            errors.append((fname, str(e)))
            print(f"\n  ERROR on {fname}: {e}")

        if (i + 1) % 100 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            eta = (len(remaining) - i - 1) / rate / 60
            print(f"  [{i+1}/{len(remaining)}] {rate:.1f} files/sec, ETA: {eta:.0f} min")

    elapsed = time.time() - start_time
    print(f"  Done: {len(remaining) - len(errors)} files in {elapsed/60:.1f} min")
    if errors:
        all_errors.extend([(output_folder, f, e) for f, e in errors])

print(f"\n{'='*60}")
print(f"  ALL RUNS COMPLETE")
if all_errors:
    print(f"  Total errors: {len(all_errors)}")
    for folder, f, e in all_errors:
        print(f"    {folder}/{f}: {e}")
else:
    print(f"  No errors!")
print(f"{'='*60}")


In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/librimix_3000'

PAIRS = [
    ('mix_both', 'after_reuse_mix'),
    ('s1', 'after_reuse_s1'),
    ('s1_noisy_40', 'after_reuse_s1_noisy_40'),
]

print(f"{'Input':<20} {'Count':>6}   {'Output':<30} {'Count':>6}   {'Status'}")
print("-" * 95)

all_done = True
for inp, out in PAIRS:
    inp_path = os.path.join(DRIVE_ROOT, inp)
    out_path = os.path.join(DRIVE_ROOT, out)
    n_in = len([f for f in os.listdir(inp_path) if f.endswith('.wav')]) if os.path.exists(inp_path) else 0
    n_out = len([f for f in os.listdir(out_path) if f.endswith('.wav')]) if os.path.exists(out_path) else 0
    ok = "DONE" if n_in == n_out else f"MISSING {n_in - n_out}"
    if n_in != n_out:
        all_done = False
    print(f"{inp:<20} {n_in:>6}   {out:<30} {n_out:>6}   {ok}")

print()
if all_done:
    print("All 3 runs complete! Download these folders to your Mac:")
    print("  1. after_reuse_mix/")
    print("  2. after_reuse_s1/")
    print("  3. after_reuse_s1_noisy_40/")
    print()
    print("You already have locally:")
    print("  - s1/ at Desktop/Libri2Mix_Test_Full/sep_clean/s1/")
    print("  - mix_both/ at Desktop/Libri2Mix_Test_Full/sep_clean/mix_both/")
    print("  - s1_noisy_40/ at AD-FlowTSE/reuse_libri1mix_experiment/input/s1_noisy_40/")
else:
    print("Some runs incomplete. Re-run Cell 7 to resume (it skips finished files).")


In [ ]:
!pip install -q transformers safetensors loralib

import os
if not os.path.exists('./vox-profile-release'):
    !git clone https://huggingface.co/tiantiaf/whisper-large-v3-voice-quality ./vox-profile-release
    print("Cloned vox-profile-release")
else:
    print("vox-profile-release already exists")

from huggingface_hub import snapshot_download
snapshot_download(repo_id="tiantiaf/whisper-large-v3-voice-quality",
                  local_dir="./vox-profile-release",
                  local_dir_use_symlinks=False)
print("Voice quality model weights downloaded.")
